In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("data/sales_data_wb.csv")


In [3]:
import pandas as pd

df['OrderDate'] = pd.to_datetime(
    df['OrderDate'],
    errors='coerce'
)

* Check Date Quality

In [4]:
df['OrderDate'].isna().sum()
df = df.dropna(subset=['OrderDate'])

* Step 2: Convert Numeric Columns

In [5]:
money_cols = [
    'OrderValue',
    'DiscountValue',
    'SalesValue',
    'SalesReturn',
    'CreditedAmount',
    'Commission'
]

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .replace('nan', 0)
        .astype(float)
    )


df[money_cols].dtypes

OrderValue        float64
DiscountValue     float64
SalesValue        float64
SalesReturn       float64
CreditedAmount    float64
Commission        float64
dtype: object

* Step 3: Create Customer Master

In [6]:
customers = (
    df['CustomerName']
    .dropna()
    .drop_duplicates()
    .sort_values()
)

customer_master = pd.DataFrame({
    'CustomerID': [f'CUST{i:04d}' for i in range(1, len(customers)+1)],
    'CustomerName': customers.values
})

* Step 4: Merge Customer IDs Back

In [7]:
df = df.merge(
    customer_master,
    on='CustomerName',
    how='left',
    suffixes=('', '_new')
)

df['CustomerID'] = df['CustomerID_new']

df.drop(columns=['CustomerID_new'], inplace=True)

* Auto Generate Invoice ID

In [8]:
missing_mask = df['InvoiceNo'].isna()

df.loc[missing_mask, 'InvoiceNo'] = ('INV' + (df[missing_mask].index + 1).astype(str).str.zfill(6))

In [9]:
df

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,CustomerID,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks
0,2025-01-01,EMWS001,Al Amin,2024,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.00,NaN
1,2025-03-01,EMWS001,Al Amin,333,Rahaman Corporation,CUST0039,0.0,0.0,0.0,0.0,25000.0,0.00,NaN
2,2025-05-01,EMWS001,Al Amin,INV000003,Popular Alumium,CUST0038,0.0,0.0,0.0,0.0,44300.0,0.00,NaN
3,2025-06-01,EMWS001,Al Amin,INV000004,Silvia Crockerise,CUST0050,0.0,0.0,0.0,0.0,21000.0,0.00,NaN
4,2025-08-01,EMWS001,Al Amin,17,Supper Pavillion,CUST0051,16175.5,0.0,16175.5,0.0,0.0,0.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2025-05-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,CUST0028,0.0,0.0,0.0,0.0,128576.0,2571.52,NaN
116,2025-05-03,EMWS008,Yousuf Mazumder Anik,113,Ma Crockerise Shop,CUST0030,0.0,0.0,0.0,0.0,1424.0,0.00,NaN
117,2025-11-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,CUST0028,0.0,0.0,0.0,0.0,70700.0,1414.00,NaN
118,2025-07-04,EMWS008,Yousuf Mazumder Anik,ORD0000011,Kawchar Store,CUST0028,45362.4,0.0,45362.4,0.0,0.0,0.00,NaN


In [10]:
df.describe()

,OrderDate,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks
count,120,120.000000,120.000000,120.000000,120.000000,120.000000,120.000000,0.0
mean,2025-06-22 12:48:00,21058.539667,75.159583,20983.380083,132.990833,11026.941667,230.005167,NaN
min,2025-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,2025-04-02 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,2025-07-05 00:00:00,0.000000,0.000000,0.000000,0.000000,1712.000000,0.000000,NaN
75%,2025-09-04 00:00:00,16992.350000,0.000000,16632.375000,0.000000,6864.500000,0.000000,NaN
max,2025-12-04 00:00:00,719172.800000,3702.600000,719172.800000,15958.900000,250000.000000,7000.000000,NaN
std,NaN,72226.513136,388.405716,72155.044702,1456.841587,30121.088912,1048.298344,NaN


In [11]:
df.isnull().sum()

OrderDate           0
SalesID             0
SalesName           0
InvoiceNo           0
CustomerName        4
CustomerID          4
OrderValue          0
DiscountValue       0
SalesValue          0
SalesReturn         0
CreditedAmount      0
Commission          0
Remarks           120
dtype: int64

In [12]:
df['CustomerName'] = df.groupby('InvoiceNo')['CustomerName'].transform(
    lambda x: x.ffill().bfill()
)

In [13]:
df['CustomerName'] = df['CustomerName'].fillna('UNKNOWN CUSTOMER')

In [14]:
df.isnull().sum()

OrderDate           0
SalesID             0
SalesName           0
InvoiceNo           0
CustomerName        0
CustomerID          4
OrderValue          0
DiscountValue       0
SalesValue          0
SalesReturn         0
CreditedAmount      0
Commission          0
Remarks           120
dtype: int64

In [15]:
df['CustomerID'] = df['CustomerName'].factorize()[0] + 1
df['CustomerID'] = 'CUST' + df['CustomerID'].astype(str).str.zfill(5)

In [16]:
df.isnull().sum()

OrderDate           0
SalesID             0
SalesName           0
InvoiceNo           0
CustomerName        0
CustomerID          0
OrderValue          0
DiscountValue       0
SalesValue          0
SalesReturn         0
CreditedAmount      0
Commission          0
Remarks           120
dtype: int64

In [17]:
df['Remarks'] = df['Remarks'].fillna('N/A')

In [18]:
df.isna().sum()

OrderDate         0
SalesID           0
SalesName         0
InvoiceNo         0
CustomerName      0
CustomerID        0
OrderValue        0
DiscountValue     0
SalesValue        0
SalesReturn       0
CreditedAmount    0
Commission        0
Remarks           0
dtype: int64

In [19]:
df

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,CustomerID,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks
0,2025-01-01,EMWS001,Al Amin,2024,Dali Super Shop,CUST00001,0.0,0.0,0.0,0.0,0.0,0.00,N/A
1,2025-03-01,EMWS001,Al Amin,333,Rahaman Corporation,CUST00002,0.0,0.0,0.0,0.0,25000.0,0.00,N/A
2,2025-05-01,EMWS001,Al Amin,INV000003,Popular Alumium,CUST00003,0.0,0.0,0.0,0.0,44300.0,0.00,N/A
3,2025-06-01,EMWS001,Al Amin,INV000004,Silvia Crockerise,CUST00004,0.0,0.0,0.0,0.0,21000.0,0.00,N/A
4,2025-08-01,EMWS001,Al Amin,17,Supper Pavillion,CUST00005,16175.5,0.0,16175.5,0.0,0.0,0.00,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2025-05-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,CUST00052,0.0,0.0,0.0,0.0,128576.0,2571.52,N/A
116,2025-05-03,EMWS008,Yousuf Mazumder Anik,113,Ma Crockerise Shop,CUST00046,0.0,0.0,0.0,0.0,1424.0,0.00,N/A
117,2025-11-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,CUST00052,0.0,0.0,0.0,0.0,70700.0,1414.00,N/A
118,2025-07-04,EMWS008,Yousuf Mazumder Anik,ORD0000011,Kawchar Store,CUST00052,45362.4,0.0,45362.4,0.0,0.0,0.00,N/A


In [20]:
total_customers = df['CustomerName'].nunique()
total_customers

54

In [21]:
customer_master = (
    df[['CustomerName']]
    .dropna()
    .drop_duplicates()
    .sort_values('CustomerName')
    .reset_index(drop=True)
)

In [22]:
customer_master['CustomerID'] = (
    'CUST' + (customer_master.index + 1).astype(str).str.zfill(5)
)

In [23]:
df = df.drop(columns=['CustomerID'], errors='ignore')

df = df.merge(customer_master, on='CustomerName', how='left')

In [24]:
customer_master.head(50)

,CustomerName,CustomerID
0,Abid Trading,CUST00001
1,Al - Aksa Crockeries,CUST00002
2,Al Madina Crockeries,CUST00003
3,Alif Crockeries,CUST00004
4,Alif Crockerise,CUST00005
5,Anonda Crockeries,CUST00006
6,Anondo Crokerise,CUST00007
7,Articuler Corporation,CUST00008
8,BANK_DP,CUST00009
9,Beli Crockerise,CUST00010


In [25]:
customer_summary = (
    df.groupby(['CustomerName','SalesName'])
    .agg({
        'SalesValue': 'sum',
        'CreditedAmount': 'sum',
        'SalesReturn': 'sum'
    })
    .reset_index()
)
print(customer_summary.head(200))

                    CustomerName             SalesName  SalesValue  \
0                   Abid Trading    Sajib Kumar Biswas        0.00   
1           Al - Aksa Crockeries               Al Amin    67371.00   
2           Al Madina Crockeries    Sajib Kumar Biswas        0.00   
3                Alif Crockeries               Al Amin     9409.50   
4                Alif Crockerise               Al Amin        0.00   
5              Anonda Crockeries               Al Amin    19048.50   
6               Anondo Crokerise               Al Amin        0.00   
7          Articuler Corporation  Yousuf Mazumder Anik     2184.00   
8                        BANK_DP    Sajib Kumar Biswas        0.00   
9                Beli Crockerise               Al Amin        0.00   
10   Bhai Bhai Crockeries (2024)    Sajib Kumar Biswas        0.00   
11       Bismillallah Crockeries    Sajib Kumar Biswas        0.00   
12         Bismilliah Crockeries    Sajib Kumar Biswas    15563.00   
13  Bismilliah Crock

In [26]:
customer_master = (
    df[['CustomerName']]
    .dropna()
    .drop_duplicates()
    .sort_values('CustomerName')
    .reset_index(drop=True)
)

customer_master['CustomerID'] = (
    'CUST' +
    (customer_master.index + 1).astype(str).str.zfill(5)
)

print(customer_master.head())

           CustomerName CustomerID
0          Abid Trading  CUST00001
1  Al - Aksa Crockeries  CUST00002
2  Al Madina Crockeries  CUST00003
3       Alif Crockeries  CUST00004
4       Alif Crockerise  CUST00005


In [27]:
df = df.drop(columns=['CustomerID'], errors='ignore')

df = df.merge(
    customer_master,
    on='CustomerName',
    how='left'
)

In [28]:
customer_summary = (
    df.groupby(['CustomerID', 'CustomerName', 'SalesName'])
    .agg({
        'SalesValue': 'sum',
        'CreditedAmount': 'sum',
        'SalesReturn': 'sum'
    })
    .reset_index()
)

print(customer_summary.head())

  CustomerID          CustomerName           SalesName  SalesValue  \
0  CUST00001          Abid Trading  Sajib Kumar Biswas         0.0   
1  CUST00002  Al - Aksa Crockeries             Al Amin     67371.0   
2  CUST00003  Al Madina Crockeries  Sajib Kumar Biswas         0.0   
3  CUST00004       Alif Crockeries             Al Amin      9409.5   
4  CUST00005       Alif Crockerise             Al Amin         0.0   

   CreditedAmount  SalesReturn  
0          4250.0          0.0  
1          9500.0          0.0  
2          5164.0          0.0  
3          5400.0          0.0  
4          3310.0          0.0  


In [29]:
print("Total Unique Customers:", customer_master['CustomerID'].nunique())

Total Unique Customers: 54


In [30]:
output_csv = "data\\clean_sales_data.csv"

df.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

print("CSV Saved:", output_csv)

CSV Saved: data\clean_sales_data.csv


In [31]:
output_excel = "data\\clean_sales_data.xlsx"

df.to_excel(
    output_excel,
    index=False
)

print("Excel Saved:", output_excel)

Excel Saved: data\clean_sales_data.xlsx


In [32]:
df = df.columns.tolist()
df

['OrderDate',
 'SalesID',
 'SalesName',
 'InvoiceNo',
 'CustomerName',
 'OrderValue',
 'DiscountValue',
 'SalesValue',
 'SalesReturn',
 'CreditedAmount',
 'Commission',
 'Remarks',
 'CustomerID']

In [33]:
import pandas as pd
df = "data/clean_sales_data.csv"
df = pd.read_csv(df)
df

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks,CustomerID
0,2025-01-01,EMWS001,Al Amin,2024,Dali Super Shop,0.0,0.0,0.0,0.0,0.0,0.00,NaN,CUST00016
1,2025-03-01,EMWS001,Al Amin,333,Rahaman Corporation,0.0,0.0,0.0,0.0,25000.0,0.00,NaN,CUST00039
2,2025-05-01,EMWS001,Al Amin,INV000003,Popular Alumium,0.0,0.0,0.0,0.0,44300.0,0.00,NaN,CUST00038
3,2025-06-01,EMWS001,Al Amin,INV000004,Silvia Crockerise,0.0,0.0,0.0,0.0,21000.0,0.00,NaN,CUST00050
4,2025-08-01,EMWS001,Al Amin,17,Supper Pavillion,16175.5,0.0,16175.5,0.0,0.0,0.00,NaN,CUST00051
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2025-05-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,0.0,0.0,0.0,0.0,128576.0,2571.52,NaN,CUST00028
116,2025-05-03,EMWS008,Yousuf Mazumder Anik,113,Ma Crockerise Shop,0.0,0.0,0.0,0.0,1424.0,0.00,NaN,CUST00030
117,2025-11-03,EMWS008,Yousuf Mazumder Anik,153,Kawchar Store,0.0,0.0,0.0,0.0,70700.0,1414.00,NaN,CUST00028
118,2025-07-04,EMWS008,Yousuf Mazumder Anik,ORD0000011,Kawchar Store,45362.4,0.0,45362.4,0.0,0.0,0.00,NaN,CUST00028


##### Dataset Overview

In [34]:
print(df.shape)
print(df.info())
print(df.describe())

(120, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       120 non-null    object 
 1   SalesID         120 non-null    object 
 2   SalesName       120 non-null    object 
 3   InvoiceNo       120 non-null    object 
 4   CustomerName    120 non-null    object 
 5   OrderValue      120 non-null    float64
 6   DiscountValue   120 non-null    float64
 7   SalesValue      120 non-null    float64
 8   SalesReturn     120 non-null    float64
 9   CreditedAmount  120 non-null    float64
 10  Commission      120 non-null    float64
 11  Remarks         0 non-null      float64
 12  CustomerID      120 non-null    object 
dtypes: float64(7), object(6)
memory usage: 12.3+ KB
None
          OrderValue  DiscountValue     SalesValue   SalesReturn  \
count     120.000000     120.000000     120.000000    120.000000   
mean    21058.53

##### Key Questions
* How many transactions?
* How many customers?
* How many sales executives?
* Total sales amount?

In [35]:
print("Total Customers:", df['CustomerID'].nunique())
print("Total Sales Executives:", df['SalesName'].nunique())
print("Total Sales:", df['SalesValue'].sum())

Total Customers: 54
Total Sales Executives: 3
Total Sales: 2518005.6100000003


In [36]:
df = pd.read_csv("data/sales_data_wb.csv")
df

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,CustomerID,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks
0,01/01/2025,EMWS001,Al Amin,2024,NaN,NaN,0.00,0.00,0.00,NaN,0.00,NaN,NaN
1,03/01/2025,EMWS001,Al Amin,333,Rahaman Corporation,NaN,0.00,0.00,0.00,NaN,"25,000.00",NaN,NaN
2,05/01/2025,EMWS001,Al Amin,NaN,Popular Alumium,NaN,0.00,0.00,0.00,NaN,"44,300.00",NaN,NaN
3,06/01/2025,EMWS001,Al Amin,NaN,Silvia Crockerise,NaN,0.00,0.00,0.00,NaN,"21,000.00",NaN,NaN
4,08/01/2025,EMWS001,Al Amin,17,Supper Pavillion,NaN,"16,175.50",0.00,"16,175.50",NaN,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,31-07-2025,EMWS008,Yousuf Mazumder Anik,NaN,Friends Crockeries,NaN,0.00,0.00,0.00,0.00,"18,600.00",0.00,NaN
878,07-08-2025,EMWS008,Yousuf Mazumder Anik,ORD0000162,Grameen Crockeries,NaN,0.00,0.00,0.00,0.00,"5,450.00",0.00,NaN
879,20-08-2025,EMWS008,Yousuf Mazumder Anik,NaN,Mohammadia Trading,NaN,0.00,0.00,0.00,0.00,0.00,0.00,NaN
880,20-08-2025,EMWS008,Yousuf Mazumder Anik,NaN,Raisa Store,NaN,0.00,0.00,0.00,0.00,0.00,0.00,NaN


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       882 non-null    object 
 1   SalesID         882 non-null    object 
 2   SalesName       882 non-null    object 
 3   InvoiceNo       477 non-null    object 
 4   CustomerName    875 non-null    object 
 5   CustomerID      0 non-null      float64
 6   OrderValue      882 non-null    object 
 7   DiscountValue   874 non-null    object 
 8   SalesValue      882 non-null    object 
 9   SalesReturn     779 non-null    object 
 10  CreditedAmount  881 non-null    object 
 11  Commission      753 non-null    object 
 12  Remarks         0 non-null      float64
dtypes: float64(2), object(11)
memory usage: 89.7+ KB


| Column         | Problem                     |
| -------------- | --------------------------- |
| OrderDate      | object → should be datetime |
| InvoiceNo      | 405 missing                 |
| CustomerName   | 7 missing                   |
| CustomerID     | Completely empty            |
| OrderValue     | object with commas          |
| DiscountValue  | 8 missing + object          |
| SalesValue     | object with commas          |
| SalesReturn    | 103 missing                 |
| CreditedAmount | 1 missing                   |
| Commission     | 129 missing                 |
| Remarks        | Completely empty            |


#### Step 1: Convert Date

In [38]:
# Keep as string first to check for parsing issues:
df['OrderDate'] = df['OrderDate'].astype(str)

In [39]:
#Standardize separators
df['OrderDate'] = (
    df['OrderDate']
    .str.replace('-', '/', regex=False)
    .str.replace('.', '/', regex=False)
)

In [40]:
# Convert to datetime (SMART parsing)

df['OrderDate'] = pd.to_datetime(
    df['OrderDate'],
    errors='coerce',
    dayfirst=True
)

In [41]:
df['OrderDate'] = df['OrderDate'].apply(
    lambda x: x.replace(year=x.year + 100) if x.year < 2000 else x
)

In [42]:
# Check broken dates
df[df['OrderDate'].isna()]

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,CustomerID,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks
248,NaT,EMWS003,Sajib Kumar Biswas,NaN,Jui Crockeries,NaN,0.00,0.00,0.00,0.00,"250,000.00","17,500.00",NaN
463,NaT,EMWS005,Mynuddin Hasan Hridoy,MHH01,Promotional Sample,NaN,"2,460.75",0.00,"2,460.75",0.00,0.00,0.00,NaN


In [43]:
df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.month_name()
df['Day'] = df['OrderDate'].dt.day

##### Step 2: Convert Money Columns

In [44]:
money_cols = [
    'OrderValue',
    'DiscountValue',
    'SalesValue',
    'SalesReturn',
    'CreditedAmount',
    'Commission'
]

for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .replace('nan', '0')
        .replace('', '0')
        .astype(float)
    )

##### Step 3: Fill Missing Numeric Values

In [45]:
df['DiscountValue'] = df['DiscountValue'].fillna(0)
df['SalesReturn'] = df['SalesReturn'].fillna(0)
df['CreditedAmount'] = df['CreditedAmount'].fillna(0)
df['Commission'] = df['Commission'].fillna(0)

#### Step 4: Clean Customer Names
* Remove leading/trailing spaces:

In [46]:
df['CustomerName'] = (
    df['CustomerName']
    .astype(str)
    .str.strip()
    .str.title()
)

#### Find missing customers:

In [47]:
df[df['CustomerName'].isna()]

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,CustomerID,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Remarks,Year,Month,Day


In [48]:
# Fill Customer
df['CustomerName'] = df['CustomerName'].fillna(
    'Unknown Customer'
)

#### Step 5: Create Customer Master

* Generate unique customer codes:

In [49]:
customer_master = (
    df[['CustomerName']]
    .drop_duplicates()
    .sort_values('CustomerName')
    .reset_index(drop=True)
)

customer_master['CustomerID'] = (
    'CUST' +
    (customer_master.index + 1)
    .astype(str)
    .str.zfill(5)
)

#### Step 6: Merge Customer Codes

In [50]:
df = df.drop(columns=['CustomerID'])

df = df.merge(
    customer_master,
    on='CustomerName',
    how='left'
)

#### Step 7: Fix Invoice Numbers

* Replace missing and numeric-only invoices.

In [51]:
import pandas as pd

def clean_invoice(x):
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.isdigit():
        return None

    return x

df['InvoiceNo'] = df['InvoiceNo'].apply(clean_invoice)

# Generate IDs:
mask = df['InvoiceNo'].isna()

df.loc[mask, 'InvoiceNo'] = [
    f'INV{str(i).zfill(6)}'
    for i in range(1, mask.sum()+1)
]

#### Step 8: Remove Empty Remarks Column

* Since all 882 values are empty:

In [52]:
df.drop(columns=['Remarks'], inplace=True)

#### Step 9: Create Analytics Columns

In [53]:
df['NetSales'] = (
    df['SalesValue']
    - df['SalesReturn']
)

df['DueAmount'] = (
    df['NetSales']
    - df['CreditedAmount']
)

df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.month_name()

##### Step 10: Save Clean Files

In [54]:
df.to_excel(
    'data/new_clean_sales_data.xlsx',
    index=False
)

df.to_csv(
    'data/new_clean_sales_data.csv',
    index=False,
    encoding='utf-8-sig'
)

customer_master.to_excel(
    'data/customer_master.xlsx',
    index=False
)

In [55]:
import pandas as pd
df = pd.read_csv("data/new_clean_sales_data.csv")
df

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,Month,Day,CustomerID,NetSales,DueAmount
0,2025-01-01,EMWS001,Al Amin,INV000001,Nan,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,January,1.0,CUST00115,0.0,0.0
1,2025-01-03,EMWS001,Al Amin,INV000002,Rahaman Corporation,0.0,0.0,0.0,0.0,25000.0,0.0,2025.0,January,3.0,CUST00131,0.0,-25000.0
2,2025-01-05,EMWS001,Al Amin,INV000003,Popular Alumium,0.0,0.0,0.0,0.0,44300.0,0.0,2025.0,January,5.0,CUST00127,0.0,-44300.0
3,2025-01-06,EMWS001,Al Amin,INV000004,Silvia Crockerise,0.0,0.0,0.0,0.0,21000.0,0.0,2025.0,January,6.0,CUST00165,0.0,-21000.0
4,2025-01-08,EMWS001,Al Amin,INV000005,Supper Pavillion,16175.5,0.0,16175.5,0.0,0.0,0.0,2025.0,January,8.0,CUST00170,16175.5,16175.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,2025-07-31,EMWS008,Yousuf Mazumder Anik,INV000615,Friends Crockeries,0.0,0.0,0.0,0.0,18600.0,0.0,2025.0,July,31.0,CUST00060,0.0,-18600.0
878,2025-08-07,EMWS008,Yousuf Mazumder Anik,ORD0000162,Grameen Crockeries,0.0,0.0,0.0,0.0,5450.0,0.0,2025.0,August,7.0,CUST00063,0.0,-5450.0
879,2025-08-20,EMWS008,Yousuf Mazumder Anik,INV000616,Mohammadia Trading,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,August,20.0,CUST00105,0.0,0.0
880,2025-08-20,EMWS008,Yousuf Mazumder Anik,INV000617,Raisa Store,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,August,20.0,CUST00134,0.0,0.0


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       880 non-null    object 
 1   SalesID         882 non-null    object 
 2   SalesName       882 non-null    object 
 3   InvoiceNo       882 non-null    object 
 4   CustomerName    882 non-null    object 
 5   OrderValue      882 non-null    float64
 6   DiscountValue   882 non-null    float64
 7   SalesValue      882 non-null    float64
 8   SalesReturn     882 non-null    float64
 9   CreditedAmount  882 non-null    float64
 10  Commission      882 non-null    float64
 11  Year            880 non-null    float64
 12  Month           880 non-null    object 
 13  Day             880 non-null    float64
 14  CustomerID      882 non-null    object 
 15  NetSales        882 non-null    float64
 16  DueAmount       882 non-null    float64
dtypes: float64(10), object(7)
memory us

In [57]:
df.shape

(882, 17)

#### Step 1: Check missing dates properly

In [58]:
df[df['OrderDate'].isna()].head(20)

df['OrderDate'].astype(str).value_counts().head(20)

OrderDate
2025-06-01    25
2025-06-30    23
2025-10-05    18
2025-09-14    14
2025-08-20    14
2025-08-24    13
2025-10-14    13
2025-05-31    12
2025-09-18    12
2025-08-10    12
2025-09-03    11
2025-02-17    11
2025-09-10    11
2025-09-04    11
2025-09-21    11
2025-09-25    10
2025-02-19    10
2025-07-31    10
2025-06-18    10
2025-02-10    10
Name: count, dtype: int64

In [59]:
# Fill from Invoice grouping (if invoice has pattern)

df['OrderDate'] = df.groupby('InvoiceNo')['OrderDate'].transform(
    lambda x: x.ffill().bfill()
)

# Fill from Sales Executive pattern (fallback)
df['OrderDate'] = df.groupby('SalesID')['OrderDate'].transform(
    lambda x: x.ffill().bfill()
)

#If still missing → mark safely as 'Unknown Date'
df['OrderDate'] = df['OrderDate'].fillna(pd.Timestamp('1900-01-01'))



In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       882 non-null    object 
 1   SalesID         882 non-null    object 
 2   SalesName       882 non-null    object 
 3   InvoiceNo       882 non-null    object 
 4   CustomerName    882 non-null    object 
 5   OrderValue      882 non-null    float64
 6   DiscountValue   882 non-null    float64
 7   SalesValue      882 non-null    float64
 8   SalesReturn     882 non-null    float64
 9   CreditedAmount  882 non-null    float64
 10  Commission      882 non-null    float64
 11  Year            880 non-null    float64
 12  Month           880 non-null    object 
 13  Day             880 non-null    float64
 14  CustomerID      882 non-null    object 
 15  NetSales        882 non-null    float64
 16  DueAmount       882 non-null    float64
dtypes: float64(10), object(7)
memory us

In [61]:
df[df['Day'].isna()].head(20)

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,Month,Day,CustomerID,NetSales,DueAmount
248,2025-02-24,EMWS003,Sajib Kumar Biswas,INV000226,Jui Crockeries,0.00,0.0,0.00,0.0,250000.0,17500.0,NaN,NaN,NaN,CUST00076,0.00,-250000.00
463,2025-06-01,EMWS005,Mynuddin Hasan Hridoy,MHH01,Promotional Sample,2460.75,0.0,2460.75,0.0,0.0,0.0,NaN,NaN,NaN,CUST00128,2460.75,2460.75


In [62]:
# Fill from Invoice grouping (if invoice has pattern)

df['Day'] = df.groupby('InvoiceNo')['OrderDate'].transform(
    lambda x: x.ffill().bfill()
)

# Fill from Sales Executive pattern (fallback)
df['Day'] = df.groupby('SalesID')['OrderDate'].transform(
    lambda x: x.ffill().bfill()
)

#If still missing → mark safely as 'Unknown Date'
df['Day'] = df['OrderDate'].fillna(pd.Timestamp('1900-01-01'))

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       882 non-null    object 
 1   SalesID         882 non-null    object 
 2   SalesName       882 non-null    object 
 3   InvoiceNo       882 non-null    object 
 4   CustomerName    882 non-null    object 
 5   OrderValue      882 non-null    float64
 6   DiscountValue   882 non-null    float64
 7   SalesValue      882 non-null    float64
 8   SalesReturn     882 non-null    float64
 9   CreditedAmount  882 non-null    float64
 10  Commission      882 non-null    float64
 11  Year            880 non-null    float64
 12  Month           880 non-null    object 
 13  Day             882 non-null    object 
 14  CustomerID      882 non-null    object 
 15  NetSales        882 non-null    float64
 16  DueAmount       882 non-null    float64
dtypes: float64(9), object(8)
memory usa

In [64]:
df[df['OrderDate'].isna()]

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,Month,Day,CustomerID,NetSales,DueAmount


In [65]:
df.describe()

,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,NetSales,DueAmount
count,8.820000e+02,882.000000,8.820000e+02,882.000000,882.000000,882.000000,880.0,8.820000e+02,8.820000e+02
mean,1.286010e+04,13.362483,1.322300e+04,475.496712,8778.730159,195.403333,2025.0,1.274750e+04,3.968769e+03
std,5.228356e+04,155.430504,5.292931e+04,6800.742453,23890.284517,1640.829530,0.0,5.348225e+04,6.034537e+04
min,-9.980000e+02,0.000000,-9.980000e+02,0.000000,0.000000,0.000000,2025.0,-1.780452e+05,-2.500000e+05
25%,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2025.0,0.000000e+00,-6.638750e+03
50%,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2025.0,0.000000e+00,-1.090000e+03
75%,9.095375e+03,0.000000,9.609548e+03,0.000000,6116.250000,0.000000,2025.0,9.609548e+03,9.609548e+03
max,1.001827e+06,3702.600000,1.001827e+06,178045.250000,250000.000000,38280.000000,2025.0,1.001827e+06,1.001827e+06


In [66]:
df.to_csv(
    'data/transformed_clean_sales_data_2.csv',
    index=False,
    encoding='utf-8-sig'
)

In [67]:
def new_df():
    df = pd.read_csv('data/transformed_clean_sales_data_2.csv')
    return df


df = new_df()
df.head()

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,Month,Day,CustomerID,NetSales,DueAmount
0,2025-01-01,EMWS001,Al Amin,INV000001,Nan,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,January,2025-01-01,CUST00115,0.0,0.0
1,2025-01-03,EMWS001,Al Amin,INV000002,Rahaman Corporation,0.0,0.0,0.0,0.0,25000.0,0.0,2025.0,January,2025-01-03,CUST00131,0.0,-25000.0
2,2025-01-05,EMWS001,Al Amin,INV000003,Popular Alumium,0.0,0.0,0.0,0.0,44300.0,0.0,2025.0,January,2025-01-05,CUST00127,0.0,-44300.0
3,2025-01-06,EMWS001,Al Amin,INV000004,Silvia Crockerise,0.0,0.0,0.0,0.0,21000.0,0.0,2025.0,January,2025-01-06,CUST00165,0.0,-21000.0
4,2025-01-08,EMWS001,Al Amin,INV000005,Supper Pavillion,16175.5,0.0,16175.5,0.0,0.0,0.0,2025.0,January,2025-01-08,CUST00170,16175.5,16175.5


In [68]:
df.describe()

,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,NetSales,DueAmount
count,8.820000e+02,882.000000,8.820000e+02,882.000000,882.000000,882.000000,880.0,8.820000e+02,8.820000e+02
mean,1.286010e+04,13.362483,1.322300e+04,475.496712,8778.730159,195.403333,2025.0,1.274750e+04,3.968769e+03
std,5.228356e+04,155.430504,5.292931e+04,6800.742453,23890.284517,1640.829530,0.0,5.348225e+04,6.034537e+04
min,-9.980000e+02,0.000000,-9.980000e+02,0.000000,0.000000,0.000000,2025.0,-1.780452e+05,-2.500000e+05
25%,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2025.0,0.000000e+00,-6.638750e+03
50%,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,2025.0,0.000000e+00,-1.090000e+03
75%,9.095375e+03,0.000000,9.609548e+03,0.000000,6116.250000,0.000000,2025.0,9.609548e+03,9.609548e+03
max,1.001827e+06,3702.600000,1.001827e+06,178045.250000,250000.000000,38280.000000,2025.0,1.001827e+06,1.001827e+06


In [69]:
df.shape

(882, 17)

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   OrderDate       882 non-null    object 
 1   SalesID         882 non-null    object 
 2   SalesName       882 non-null    object 
 3   InvoiceNo       882 non-null    object 
 4   CustomerName    882 non-null    object 
 5   OrderValue      882 non-null    float64
 6   DiscountValue   882 non-null    float64
 7   SalesValue      882 non-null    float64
 8   SalesReturn     882 non-null    float64
 9   CreditedAmount  882 non-null    float64
 10  Commission      882 non-null    float64
 11  Year            880 non-null    float64
 12  Month           880 non-null    object 
 13  Day             882 non-null    object 
 14  CustomerID      882 non-null    object 
 15  NetSales        882 non-null    float64
 16  DueAmount       882 non-null    float64
dtypes: float64(9), object(8)
memory usa

In [71]:
df.columns

Index(['OrderDate', 'SalesID', 'SalesName', 'InvoiceNo', 'CustomerName',
       'OrderValue', 'DiscountValue', 'SalesValue', 'SalesReturn',
       'CreditedAmount', 'Commission', 'Year', 'Month', 'Day', 'CustomerID',
       'NetSales', 'DueAmount'],
      dtype='object')

In [72]:
customer_master = (
    df[['CustomerName']]
    .dropna()
    .drop_duplicates()
    .sort_values('CustomerName')
    .reset_index(drop=True)
)

In [73]:
customer_master['CustomerID'] = (
    'CUST' +
    (customer_master.index + 1).astype(str).str.zfill(5)
)

In [74]:
df = df.drop(columns=['CustomerID'], errors='ignore')

In [75]:
df = df.merge(customer_master, on='CustomerName', how='left')

In [76]:
print("Total Unique Customers:", customer_master['CustomerID'].nunique())
print("Mapped Customers in Data:", df['CustomerID'].nunique())

Total Unique Customers: 180
Mapped Customers in Data: 180


In [77]:
df[df['CustomerID'].isna()]

,OrderDate,SalesID,SalesName,InvoiceNo,CustomerName,OrderValue,DiscountValue,SalesValue,SalesReturn,CreditedAmount,Commission,Year,Month,Day,NetSales,DueAmount,CustomerID


In [78]:
df.to_csv(
    'data/transformed_clean_sales_data_2.csv',
    index=False,
    encoding='utf-8-sig'
)